# Phase 2 RAG-SFT — Qwen2.5 + QLoRA từ phase1 merged adapter

Notebook này train tiếp theo hướng **RAG-SFT** cho chatbot lịch sử Việt Nam:

- Input chính: `all_messages.jsonl` gồm context chunk + câu hỏi + gold answer.
- Corpus phụ trợ: `all_chunk_id.jsonl` gồm toàn bộ chunk để validate chunk id và tạo inference RAG test.
- Quy trình model học:
  1. đọc câu hỏi;
  2. nhìn các chunk được cung cấp;
  3. chọn đúng `chunk_id` trong dòng `Nguồn được dùng`;
  4. trả lời dựa trên chunk, không bịa ngoài context.

Điểm quan trọng của phase2:

- **Không train tiếp trực tiếp từ checkpoint LoRA phase1.**
- Notebook sẽ load base `Qwen/Qwen2.5-3B-Instruct`, load adapter phase1 ở Drive, **merge LoRA phase1 vào base**, lưu thành merged base tạm thời, sau đó **khởi tạo LoRA adapter mới** để train phase2.
- Loss là assistant-only weighted CE, trong đó dòng chọn nguồn/chunk được weight cao hơn phần trả lời.
- Có generation metrics cho khả năng chọn chunk: source exact match, source precision/recall/F1, format rate, insufficient-context empty-source rate, Rouge-L.
- Có early stopping theo eval loss và callback lưu adapter tốt nhất theo composite generation metric.

In [1]:
# Cell 1 — Install dependencies
# Nếu Colab yêu cầu restart runtime sau khi cài, hãy restart rồi chạy lại từ Cell 2.

!pip install -q -U \
  "transformers>=4.44.0" \
  "peft>=0.12.0" \
  "accelerate>=0.33.0" \
  "bitsandbytes>=0.43.3" \
  "datasets>=2.20.0" \
  "trl>=0.9.6" \
  "scikit-learn" \
  "rouge-score" \
  "tensorboard" \
  "safetensors"

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 151.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 158.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 146.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobu

In [2]:
# Cell 2 — Mount Google Drive

import os

IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        print("Google Drive mounted.")
    except Exception as e:
        print("Không mount được Google Drive:", repr(e))
else:
    print("Không chạy trong Colab, bỏ qua mount Google Drive.")

Mounted at /content/drive
Google Drive mounted.


In [3]:
# Cell 3 — Imports

import os
import re
import gc
import json
import math
import time
import shutil
import random
import inspect
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    TrainerCallback,
    EarlyStoppingCallback,
    set_seed,
)

from peft import (
    PeftModel,
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

try:
    from rouge_score import rouge_scorer
    ROUGE_AVAILABLE = True
except Exception:
    rouge_scorer = None
    ROUGE_AVAILABLE = False

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))

torch: 2.11.0+cu128
cuda available: True
gpu: NVIDIA A100-SXM4-40GB
capability: (8, 0)


In [4]:
# Cell 4 — Global config

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Base model giống phase1 notebook.
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# Dataset trong Google Drive.
# Hãy đặt 2 file này vào folder dưới đây:
DRIVE_DATA_DIR = "/content/drive/MyDrive/vn_history_model_backups/vn_history_rag_sft_dataset"
CHUNK_FILE = f"{DRIVE_DATA_DIR}/all_chunk_id.jsonl"
MESSAGE_FILE = f"{DRIVE_DATA_DIR}/all_messages.jsonl"

# Nếu bạn upload file thủ công vào Colab session, sửa 2 dòng này, ví dụ:
# CHUNK_FILE = "/content/all_chunk_id.jsonl"
# MESSAGE_FILE = "/content/all_messages.jsonl"

# Adapter phase1 đã train.
PHASE1_ADAPTER_DIR = "/content/drive/MyDrive/vn_history_model_backups/qwen_vnhistory_phase1_best_adapter"

# Merged base tạm: base Qwen2.5 + LoRA phase1 đã merge.
# Đây KHÔNG phải adapter phase2. Nó chỉ là base khởi tạo cho phase2.
MERGED_BASE_DIR = "/content/outputs/qwen2_5_3b_vnhistory_phase1_merged_base"

# Output phase2.
RUN_NAME = "qwen2_5_3b_vnhistory_phase2_rag_qlora"
OUTPUT_DIR = f"/content/outputs/{RUN_NAME}"
BEST_METRIC_ADAPTER_DIR = f"/content/outputs/{RUN_NAME}_best_by_generation_metric"
FINAL_EXPORT_DIR = f"/content/outputs/qwen_vnhistory_phase2_rag_best_adapter"

# Backup vào Drive.
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/vn_history_model_backups"

# Sequence length.
# all_messages có context dài; 4096 thường hợp lý cho Qwen2.5-3B QLoRA.
# Nếu OOM trên T4, giảm MAX_LENGTH = 2048, batch = 1 hoặc 2.
MAX_LENGTH = 4096

# Dataset split.
TRAIN_RATIO = 0.90
EVAL_RATIO = 0.05
TEST_RATIO = 0.05

# Có thể dùng None để train full 1000 samples.
MAX_SAMPLES = None

# Weighted CE:
# Dòng "Nguồn được dùng" là phần học chọn chunk, nên weight cao hơn answer body.
SOURCE_LINE_LOSS_WEIGHT = 1.6
ANSWER_LOSS_WEIGHT = 1.0

# Training.
NUM_EPOCHS = 5
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8

LEARNING_RATE = 1.5e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05

LOGGING_STEPS = 10
EVAL_STEPS = 50
SAVE_STEPS = 50

# Eval subset trong lúc train để đỡ chậm.
EVAL_DURING_TRAIN_SIZE = 200

# Early stopping eval_loss.
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_THRESHOLD = 0.0

# Generation metric callback.
# Đây là metric quan trọng hơn cho bài toán RAG: chọn đúng chunk + format + answer.
GEN_EVAL_MAX_EXAMPLES = 80
GEN_EVAL_EVERY_N_EVALS = 1
GEN_METRIC_PATIENCE = 4
GEN_METRIC_MIN_DELTA = 1e-4
GEN_MAX_NEW_TOKENS = 256

# LoRA phase2 mới.
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

NUM_PROC = max(1, min(8, os.cpu_count() or 1))

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(DRIVE_BACKUP_DIR).mkdir(parents=True, exist_ok=True)

print("MODEL_ID:", MODEL_ID)
print("CHUNK_FILE:", CHUNK_FILE)
print("MESSAGE_FILE:", MESSAGE_FILE)
print("PHASE1_ADAPTER_DIR:", PHASE1_ADAPTER_DIR)
print("Effective batch size:", PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * max(1, torch.cuda.device_count()))

MODEL_ID: Qwen/Qwen2.5-3B-Instruct
CHUNK_FILE: /content/drive/MyDrive/vn_history_model_backups/vn_history_rag_sft_dataset/all_chunk_id.jsonl
MESSAGE_FILE: /content/drive/MyDrive/vn_history_model_backups/vn_history_rag_sft_dataset/all_messages.jsonl
PHASE1_ADAPTER_DIR: /content/drive/MyDrive/vn_history_model_backups/qwen_vnhistory_phase1_best_adapter
Effective batch size: 16


In [5]:
# Cell 5 — Utilities

def normalize_text(x: Any) -> str:
    if x is None:
        return ""
    x = str(x)
    x = x.replace("\r\n", "\n").replace("\r", "\n")
    x = re.sub(r"[ \t]+", " ", x)
    x = re.sub(r"\n{3,}", "\n\n", x)
    return x.strip()


def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                raise ValueError(f"JSONL parse lỗi tại {path}:{line_no}: {e}") from e
    return rows


def write_json(path: str, obj: Any):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def get_size_mb(path: str) -> float:
    p = Path(path)
    if not p.exists():
        return 0.0
    if p.is_file():
        return p.stat().st_size / 1024 / 1024
    total = 0
    for item in p.rglob("*"):
        if item.is_file():
            total += item.stat().st_size
    return total / 1024 / 1024


def require_path(path: str, hint: str = ""):
    if not os.path.exists(path):
        extra = f"\nGợi ý: {hint}" if hint else ""
        raise FileNotFoundError(f"Không tìm thấy: {path}{extra}")
    print("OK:", path, f"({get_size_mb(path):.2f} MB)")


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

In [6]:
# Cell 6 — Check files and load dataset

require_path(CHUNK_FILE, "Upload all_chunk_id.jsonl vào DRIVE_DATA_DIR hoặc sửa CHUNK_FILE.")
require_path(MESSAGE_FILE, "Upload all_messages.jsonl vào DRIVE_DATA_DIR hoặc sửa MESSAGE_FILE.")
require_path(PHASE1_ADAPTER_DIR, "Folder này cần chứa adapter_config.json và adapter_model.safetensors/bin của phase1.")

chunks = read_jsonl(CHUNK_FILE)
messages = read_jsonl(MESSAGE_FILE)

print("Number of chunks:", len(chunks))
print("Number of message samples:", len(messages))

chunk_by_id = {}
for c in chunks:
    cid = c.get("chunk_id")
    if cid:
        chunk_by_id[cid] = c

all_chunk_ids = set(chunk_by_id)

print("Unique chunk_id:", len(all_chunk_ids))
print("First chunk keys:", sorted(chunks[0].keys()))
print("First message keys:", messages[0].keys())
print("First message sample type:", messages[0].get("type"))
print("First assistant preview:")
print(messages[0]["messages"][-1]["content"][:500])

OK: /content/drive/MyDrive/vn_history_model_backups/vn_history_rag_sft_dataset/all_chunk_id.jsonl (2.00 MB)
OK: /content/drive/MyDrive/vn_history_model_backups/vn_history_rag_sft_dataset/all_messages.jsonl (9.30 MB)
OK: /content/drive/MyDrive/vn_history_model_backups/qwen_vnhistory_phase1_best_adapter (239.35 MB)
Number of chunks: 520
Number of message samples: 1000
Unique chunk_id: 511
First chunk keys: ['char_len', 'chunk_id', 'chunk_index', 'history_score', 'source', 'source_type', 'text', 'title', 'url', 'word_len']
First message keys: dict_keys(['id', 'type', 'messages'])
First message sample type: noisy_context
First assistant preview:
Nguồn được dùng: [hf_wikipedia_ngô_quyền_0000_af223d790816]

Trả lời:
Chiến thắng Bạch Đằng năm 938 gắn với Ngô Quyền. Tài liệu nêu rằng năm 938, ông lãnh đạo nhân dân đánh bại quân Nam Hán trong trận Bạch Đằng.


In [7]:
# Cell 7 — Parse RAG-SFT fields from all_messages.jsonl

SOURCE_LINE_RE = re.compile(r"Nguồn được dùng:\s*\[(.*?)\]", re.IGNORECASE | re.DOTALL)
ANSWER_SPLIT_RE = re.compile(r"Trả lời\s*:", re.IGNORECASE)
CONTEXT_ID_RE = re.compile(r"^\[([^\]]+)\]\s+(.+)$", re.MULTILINE)


def extract_question(user_text: str) -> str:
    user_text = normalize_text(user_text)
    m = re.search(r"Câu hỏi:\s*(.*?)(?:\n\s*\n\s*Tài liệu tham khảo:|$)", user_text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        return normalize_text(m.group(1))
    return user_text[:300]


def extract_context_chunk_ids(user_text: str) -> List[str]:
    ids = []
    for m in CONTEXT_ID_RE.finditer(user_text or ""):
        cid = normalize_text(m.group(1))
        if cid and cid not in ids:
            ids.append(cid)
    return ids


def extract_source_ids_from_answer(answer_text: str) -> List[str]:
    answer_text = normalize_text(answer_text)
    m = SOURCE_LINE_RE.search(answer_text)
    if not m:
        return []
    inner = normalize_text(m.group(1))
    if not inner:
        return []
    ids = [x.strip().strip("`") for x in re.split(r"[,;\s]+", inner) if x.strip()]
    out = []
    for cid in ids:
        if cid not in out:
            out.append(cid)
    return out


def extract_answer_body(answer_text: str) -> str:
    answer_text = normalize_text(answer_text)
    parts = ANSWER_SPLIT_RE.split(answer_text, maxsplit=1)
    if len(parts) == 2:
        return normalize_text(parts[1])
    return answer_text


def split_assistant_for_loss(answer_text: str) -> Tuple[str, str]:
    """Return source/format segment and answer body segment for weighted CE."""
    answer_text = normalize_text(answer_text)
    m = ANSWER_SPLIT_RE.search(answer_text)
    if not m:
        return "", answer_text
    left = answer_text[:m.end()]
    right = answer_text[m.end():]
    return normalize_text(left) + "\n", normalize_text(right)


def get_user_assistant(sample: Dict[str, Any]) -> Tuple[str, str]:
    msgs = sample.get("messages", [])
    user_text = ""
    assistant_text = ""
    for msg in msgs:
        role = msg.get("role")
        content = msg.get("content", "")
        if role == "user":
            user_text = content
        elif role == "assistant":
            assistant_text = content
    return normalize_text(user_text), normalize_text(assistant_text)


records = []
bad_rows = []

for i, sample in enumerate(messages):
    user_text, assistant_text = get_user_assistant(sample)
    context_ids = extract_context_chunk_ids(user_text)
    gold_source_ids = extract_source_ids_from_answer(assistant_text)
    question = extract_question(user_text)
    answer_body = extract_answer_body(assistant_text)

    if not user_text or not assistant_text:
        bad_rows.append((i, "missing user/assistant"))
        continue

    # Gold source ids phải nằm trong context hoặc rỗng cho insufficient.
    unknown_gold = [cid for cid in gold_source_ids if cid not in all_chunk_ids]
    if unknown_gold:
        bad_rows.append((i, f"gold ids không nằm trong all_chunk_id: {unknown_gold[:3]}"))
        continue

    records.append({
        "id": sample.get("id", f"sample_{i+1:04d}"),
        "type": sample.get("type", "unknown"),
        "user_text": user_text,
        "assistant_text": assistant_text,
        "question": question,
        "answer_body": answer_body,
        "context_chunk_ids": context_ids,
        "gold_source_ids": gold_source_ids,
        "n_context": len(context_ids),
        "n_gold_source": len(gold_source_ids),
        "user_char_len": len(user_text),
        "assistant_char_len": len(assistant_text),
    })

df = pd.DataFrame(records)
print("Valid records:", len(df))
print("Bad rows:", len(bad_rows))
print(df["type"].value_counts())
print(df[["id", "type", "n_context", "n_gold_source", "question"]].head())

# Validate phân bố chunk id trong context.
all_context_ids = sorted({cid for ids in df["context_chunk_ids"] for cid in ids})
missing_context_ids = [cid for cid in all_context_ids if cid not in all_chunk_ids]
print("Unique context ids:", len(all_context_ids))
print("Missing context ids in all_chunk_id:", len(missing_context_ids))

if bad_rows[:5]:
    print("Bad row examples:", bad_rows[:5])

assert len(df) > 0
assert len(missing_context_ids) == 0, "Có context chunk_id không tồn tại trong all_chunk_id.jsonl"

Valid records: 1000
Bad rows: 0
type
noisy_context           650
grounded_qa             200
insufficient_context    100
false_premise            50
Name: count, dtype: int64
            id           type  n_context  n_gold_source  \
0  sample_0001  noisy_context          3              1   
1  sample_0002  noisy_context          3              1   
2  sample_0003  noisy_context          3              1   
3  sample_0004  noisy_context          3              1   
4  sample_0005  noisy_context          3              1   

                                            question  
0  Chiến thắng Bạch Đằng năm 938 gắn với nhân vật...  
1   Bạch Đằng năm 938 đã đánh bại quân xâm lược nào?  
2  Ý nghĩa lịch sử của chiến thắng Bạch Đằng năm ...  
3  Sau chiến thắng Bạch Đằng, Ngô Quyền đã lập ra...  
4  Kiều Công Tiễn có vai trò gì trong bối cảnh dẫ...  
Unique context ids: 469
Missing context ids in all_chunk_id: 0


In [8]:
# Cell 8 — Stratified train/eval/test split

work_df = df.copy()

if MAX_SAMPLES is not None and MAX_SAMPLES < len(work_df):
    work_df = work_df.sample(n=MAX_SAMPLES, random_state=SEED).reset_index(drop=True)

train_df, temp_df = train_test_split(
    work_df,
    test_size=(1.0 - TRAIN_RATIO),
    random_state=SEED,
    stratify=work_df["type"],
)

relative_test_ratio = TEST_RATIO / (EVAL_RATIO + TEST_RATIO)
eval_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test_ratio,
    random_state=SEED,
    stratify=temp_df["type"],
)

train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("train/eval/test:", len(train_df), len(eval_df), len(test_df))
print("\nTrain type counts:\n", train_df["type"].value_counts())
print("\nEval type counts:\n", eval_df["type"].value_counts())
print("\nTest type counts:\n", test_df["type"].value_counts())

dataset_raw = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "eval": Dataset.from_pandas(eval_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False),
})

dataset_raw

train/eval/test: 900 50 50

Train type counts:
 type
noisy_context           585
grounded_qa             180
insufficient_context     90
false_premise            45
Name: count, dtype: int64

Eval type counts:
 type
noisy_context           33
grounded_qa             10
insufficient_context     5
false_premise            2
Name: count, dtype: int64

Test type counts:
 type
noisy_context           32
grounded_qa             10
insufficient_context     5
false_premise            3
Name: count, dtype: int64


DatasetDict({
    train: Dataset({
        features: ['id', 'type', 'user_text', 'assistant_text', 'question', 'answer_body', 'context_chunk_ids', 'gold_source_ids', 'n_context', 'n_gold_source', 'user_char_len', 'assistant_char_len'],
        num_rows: 900
    })
    eval: Dataset({
        features: ['id', 'type', 'user_text', 'assistant_text', 'question', 'answer_body', 'context_chunk_ids', 'gold_source_ids', 'n_context', 'n_gold_source', 'user_char_len', 'assistant_char_len'],
        num_rows: 50
    })
    test: Dataset({
        features: ['id', 'type', 'user_text', 'assistant_text', 'question', 'answer_body', 'context_chunk_ids', 'gold_source_ids', 'n_context', 'n_gold_source', 'user_char_len', 'assistant_char_len'],
        num_rows: 50
    })
})

In [9]:
# Cell 9 — Load tokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

IM_START = "<|im_start|>"
IM_END = "<|im_end|>"

print("pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("eos token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("im_end id:", tokenizer.convert_tokens_to_ids(IM_END))

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

pad token: <|endoftext|> 151643
eos token: <|im_end|> 151645
im_end id: 151645


In [10]:
# Cell 10 — Tokenization with assistant-only weighted labels
# Không thêm system message để khớp format all_messages.jsonl: user + assistant.

def tok(text: str) -> List[int]:
    return tokenizer(text, add_special_tokens=False)["input_ids"]


def add_segment(
    input_ids: List[int],
    labels: List[int],
    loss_weights: List[float],
    text: str,
    train_on_segment: bool,
    weight: float,
):
    ids = tok(text)
    input_ids.extend(ids)
    if train_on_segment:
        labels.extend(ids)
        loss_weights.extend([float(weight)] * len(ids))
    else:
        labels.extend([-100] * len(ids))
        loss_weights.extend([0.0] * len(ids))


def build_training_example(
    user_text: str,
    assistant_text: str,
    max_length: int = MAX_LENGTH,
) -> Dict[str, Any]:
    user_text = normalize_text(user_text)
    assistant_text = normalize_text(assistant_text)

    source_segment, answer_segment = split_assistant_for_loss(assistant_text)

    input_ids = []
    labels = []
    loss_weights = []

    # User: masked.
    add_segment(
        input_ids, labels, loss_weights,
        f"{IM_START}user\n{user_text}{IM_END}\n",
        train_on_segment=False,
        weight=0.0,
    )

    # Assistant header: masked.
    add_segment(
        input_ids, labels, loss_weights,
        f"{IM_START}assistant\n",
        train_on_segment=False,
        weight=0.0,
    )

    # Source/format: higher weight.
    if source_segment:
        add_segment(
            input_ids, labels, loss_weights,
            source_segment,
            train_on_segment=True,
            weight=SOURCE_LINE_LOSS_WEIGHT,
        )

    # Answer body: normal weight.
    if answer_segment:
        add_segment(
            input_ids, labels, loss_weights,
            answer_segment + "\n",
            train_on_segment=True,
            weight=ANSWER_LOSS_WEIGHT,
        )

    # End token: normal weight.
    add_segment(
        input_ids, labels, loss_weights,
        f"{IM_END}\n",
        train_on_segment=True,
        weight=ANSWER_LOSS_WEIGHT,
    )

    attention_mask = [1] * len(input_ids)
    too_long = len(input_ids) > max_length
    has_loss = any(x != -100 for x in labels)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "loss_weights": loss_weights,
        "length": len(input_ids),
        "too_long": too_long,
        "has_loss": has_loss,
    }


def tokenize_batch(batch):
    out = {
        "input_ids": [],
        "attention_mask": [],
        "labels": [],
        "loss_weights": [],
        "length": [],
        "too_long": [],
        "has_loss": [],
    }

    for user_text, assistant_text in zip(batch["user_text"], batch["assistant_text"]):
        item = build_training_example(user_text, assistant_text, max_length=MAX_LENGTH)
        for k in out:
            out[k].append(item[k])

    return out

In [11]:
# Cell 11 — Tokenize dataset and filter too-long samples

remove_cols = dataset_raw["train"].column_names

tokenized = dataset_raw.map(
    tokenize_batch,
    batched=True,
    num_proc=NUM_PROC,
    remove_columns=remove_cols,
    desc="Tokenizing RAG-SFT dataset",
)

before_counts = {split: len(tokenized[split]) for split in tokenized}

tokenized = tokenized.filter(
    lambda x: (not x["too_long"]) and x["has_loss"],
    num_proc=NUM_PROC,
    desc="Filtering too long / no loss rows",
)

after_counts = {split: len(tokenized[split]) for split in tokenized}
print("Before:", before_counts)
print("After :", after_counts)

for split in tokenized:
    lengths = tokenized[split]["length"]
    print(
        split,
        "n=", len(lengths),
        "min=", min(lengths),
        "p50=", int(np.percentile(lengths, 50)),
        "p90=", int(np.percentile(lengths, 90)),
        "max=", max(lengths),
    )

assert len(tokenized["train"]) > 0
assert len(tokenized["eval"]) > 0
assert len(tokenized["test"]) > 0

eval_during_train = tokenized["eval"]
if len(eval_during_train) > EVAL_DURING_TRAIN_SIZE:
    eval_during_train = eval_during_train.shuffle(seed=SEED).select(range(EVAL_DURING_TRAIN_SIZE))

print("Eval during train:", len(eval_during_train))

Tokenizing RAG-SFT dataset (num_proc=8):   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenizing RAG-SFT dataset (num_proc=8):   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenizing RAG-SFT dataset (num_proc=8):   0%|          | 0/50 [00:00<?, ? examples/s]

Filtering too long / no loss rows (num_proc=8):   0%|          | 0/900 [00:00<?, ? examples/s]

Filtering too long / no loss rows (num_proc=8):   0%|          | 0/50 [00:00<?, ? examples/s]

Filtering too long / no loss rows (num_proc=8):   0%|          | 0/50 [00:00<?, ? examples/s]

Before: {'train': 900, 'eval': 50, 'test': 50}
After : {'train': 900, 'eval': 50, 'test': 50}
train n= 900 min= 391 p50= 2553 p90= 3127 max= 3415
eval n= 50 min= 773 p50= 2354 p90= 3110 max= 3271
test n= 50 min= 856 p50= 2472 p90= 3038 max= 3226
Eval during train: 50


In [12]:
# Cell 12 — Data collator for loss_weights

@dataclass
class DataCollatorForWeightedCausalLM:
    tokenizer: Any
    pad_to_multiple_of: Optional[int] = 8

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        max_len = max(len(f["input_ids"]) for f in features)
        if self.pad_to_multiple_of:
            max_len = int(math.ceil(max_len / self.pad_to_multiple_of) * self.pad_to_multiple_of)

        batch = {
            "input_ids": [],
            "attention_mask": [],
            "labels": [],
            "loss_weights": [],
        }

        pad_id = self.tokenizer.pad_token_id

        for f in features:
            n = len(f["input_ids"])
            pad_n = max_len - n

            batch["input_ids"].append(f["input_ids"] + [pad_id] * pad_n)
            batch["attention_mask"].append(f["attention_mask"] + [0] * pad_n)
            batch["labels"].append(f["labels"] + [-100] * pad_n)
            batch["loss_weights"].append(f["loss_weights"] + [0.0] * pad_n)

        return {
            "input_ids": torch.tensor(batch["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(batch["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(batch["labels"], dtype=torch.long),
            "loss_weights": torch.tensor(batch["loss_weights"], dtype=torch.float32),
        }


data_collator = DataCollatorForWeightedCausalLM(tokenizer=tokenizer)

In [13]:
!pip uninstall -y torchao
!pip install -U "torchao>=0.16.0" peft transformers accelerate

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.0 MB/s eta 0:00:00


In [14]:
# Cell 13 — Merge phase1 LoRA adapter into base, then reload merged base in 4-bit
# Quan trọng:
# - Phase1 adapter được merge vào base model.
# - Sau đó adapter phase2 mới sẽ được khởi tạo bằng get_peft_model.
# - Như vậy phase2 KHÔNG train tiếp trực tiếp từ checkpoint LoRA phase1.

major_cc = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
USE_BF16 = torch.cuda.is_available() and major_cc >= 8
compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16

print("compute_dtype:", compute_dtype)
print("MERGED_BASE_DIR:", MERGED_BASE_DIR)

if not os.path.exists(os.path.join(MERGED_BASE_DIR, "config.json")):
    print("Chưa có merged base. Bắt đầu merge phase1 adapter vào base model...")

    cleanup_cuda()

    base_for_merge = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=compute_dtype,
        device_map="auto",
        trust_remote_code=True,
    )
    base_for_merge.config.use_cache = False

    phase1_model = PeftModel.from_pretrained(
        base_for_merge,
        PHASE1_ADAPTER_DIR,
        is_trainable=False,
    )

    print("Merging phase1 LoRA into base...")
    merged_model = phase1_model.merge_and_unload()
    merged_model.config.use_cache = False

    Path(MERGED_BASE_DIR).mkdir(parents=True, exist_ok=True)
    merged_model.save_pretrained(
        MERGED_BASE_DIR,
        safe_serialization=True,
        max_shard_size="2GB",
    )
    tokenizer.save_pretrained(MERGED_BASE_DIR)

    del phase1_model, base_for_merge, merged_model
    cleanup_cuda()

    print("Saved merged base:", MERGED_BASE_DIR)
else:
    print("Đã có merged base, bỏ qua bước merge.")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MERGED_BASE_DIR,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=compute_dtype,
)

model.config.use_cache = False

print("Loaded 4-bit merged base for phase2 training.")

compute_dtype: torch.bfloat16
MERGED_BASE_DIR: /content/outputs/qwen2_5_3b_vnhistory_phase1_merged_base
Chưa có merged base. Bắt đầu merge phase1 adapter vào base model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]

Saved merged base: /content/outputs/qwen2_5_3b_vnhistory_phase1_merged_base


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loaded 4-bit merged base for phase2 training.


In [15]:
# Cell 14 — Prepare fresh phase2 QLoRA adapter

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

phase2_lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, phase2_lora_config)

print("Phase2 LoRA adapter initialized from scratch.")
model.print_trainable_parameters()

Phase2 LoRA adapter initialized from scratch.
trainable params: 59,867,136 || all params: 3,145,805,824 || trainable%: 1.9031


In [16]:
# Cell 15 — Custom Trainer with weighted cross entropy

class WeightedCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss_weights = inputs.pop("loss_weights")

        outputs = model(**inputs)
        logits = outputs.logits
        labels = inputs["labels"]

        # Causal LM shift:
        # logits[:, t] predicts labels[:, t+1].
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        shift_weights = loss_weights[:, 1:].contiguous()

        vocab_size = shift_logits.size(-1)

        flat_logits = shift_logits.view(-1, vocab_size)
        flat_labels = shift_labels.view(-1)
        flat_weights = shift_weights.view(-1)

        token_loss = F.cross_entropy(
            flat_logits,
            flat_labels,
            reduction="none",
            ignore_index=-100,
        )

        active = flat_labels.ne(-100)
        active_loss = token_loss[active]
        active_weights = flat_weights[active]

        loss = (active_loss * active_weights).sum() / active_weights.sum().clamp(min=1.0)

        return (loss, outputs) if return_outputs else loss

In [17]:
# Cell 16 — Generation metrics for RAG chunk selection

def build_prompt_from_user_text(user_text: str) -> str:
    user_text = normalize_text(user_text)
    return f"{IM_START}user\n{user_text}{IM_END}\n{IM_START}assistant\n"


def get_model_device(model) -> torch.device:
    try:
        return model.get_input_embeddings().weight.device
    except Exception:
        return next(model.parameters()).device


@torch.inference_mode()
def generate_from_user_text(
    model,
    tokenizer,
    user_text: str,
    max_new_tokens: int = GEN_MAX_NEW_TOKENS,
    temperature: float = 0.0,
    top_p: float = 1.0,
) -> str:
    model.eval()
    prompt = build_prompt_from_user_text(user_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    device = get_model_device(model)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    im_end_id = tokenizer.convert_tokens_to_ids(IM_END)
    eos_ids = [tokenizer.eos_token_id]
    if isinstance(im_end_id, int) and im_end_id >= 0:
        eos_ids.append(im_end_id)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else None,
        top_p=top_p,
        eos_token_id=eos_ids,
        pad_token_id=tokenizer.pad_token_id,
    )

    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated, skip_special_tokens=False)
    text = text.replace(IM_END, "")
    if tokenizer.eos_token:
        text = text.replace(tokenizer.eos_token, "")
    return normalize_text(text)


def set_metrics(pred_ids: List[str], gold_ids: List[str]) -> Dict[str, float]:
    p = set(pred_ids)
    g = set(gold_ids)

    if not p and not g:
        return {
            "source_exact": 1.0,
            "source_precision": 1.0,
            "source_recall": 1.0,
            "source_f1": 1.0,
        }

    exact = float(p == g)
    tp = len(p & g)
    precision = tp / len(p) if p else 0.0
    recall = tp / len(g) if g else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "source_exact": exact,
        "source_precision": precision,
        "source_recall": recall,
        "source_f1": f1,
    }


def rouge_l_f1(pred_answer: str, gold_answer: str) -> float:
    if not ROUGE_AVAILABLE:
        return 0.0
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    return float(scorer.score(gold_answer, pred_answer)["rougeL"].fmeasure)


def compute_generation_metrics(
    model,
    tokenizer,
    eval_records: List[Dict[str, Any]],
    max_examples: int = GEN_EVAL_MAX_EXAMPLES,
    verbose: bool = False,
) -> Tuple[Dict[str, float], List[Dict[str, Any]]]:
    if max_examples is not None and len(eval_records) > max_examples:
        # deterministic subset for comparable metrics.
        rng = random.Random(SEED)
        eval_records = rng.sample(eval_records, max_examples)

    rows = []
    metric_rows = []

    start = time.time()

    for idx, rec in enumerate(eval_records):
        pred = generate_from_user_text(
            model=model,
            tokenizer=tokenizer,
            user_text=rec["user_text"],
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            temperature=0.0,
            top_p=1.0,
        )

        pred_ids = extract_source_ids_from_answer(pred)
        gold_ids = rec["gold_source_ids"]
        context_ids = rec["context_chunk_ids"]

        pred_answer_body = extract_answer_body(pred)
        gold_answer_body = rec["answer_body"]

        sm = set_metrics(pred_ids, gold_ids)

        format_ok = float(
            pred.strip().startswith("Nguồn được dùng:")
            and bool(ANSWER_SPLIT_RE.search(pred))
        )
        answer_nonempty = float(len(pred_answer_body.strip()) >= 10)
        pred_ids_in_context = float(all(cid in set(context_ids) for cid in pred_ids))
        pred_ids_in_corpus = float(all(cid in all_chunk_ids for cid in pred_ids))

        insufficient_empty = None
        if rec["type"] == "insufficient_context":
            insufficient_empty = float(pred_ids == [])

        metric_rows.append({
            **sm,
            "format_ok": format_ok,
            "answer_nonempty": answer_nonempty,
            "pred_ids_in_context": pred_ids_in_context,
            "pred_ids_in_corpus": pred_ids_in_corpus,
            "rouge_l_f1": rouge_l_f1(pred_answer_body, gold_answer_body),
            "insufficient_empty": insufficient_empty,
        })

        rows.append({
            "id": rec["id"],
            "type": rec["type"],
            "question": rec["question"],
            "gold_ids": gold_ids,
            "pred_ids": pred_ids,
            "gold_answer": rec["assistant_text"],
            "pred_answer": pred,
        })

        if verbose and idx < 3:
            print("=" * 100)
            print("QUESTION:", rec["question"])
            print("GOLD IDS:", gold_ids)
            print("PRED IDS:", pred_ids)
            print("PRED:", pred[:1000])

    def mean_key(key: str) -> float:
        vals = [r[key] for r in metric_rows if r.get(key) is not None]
        return float(np.mean(vals)) if vals else 0.0

    metrics = {
        "source_exact": mean_key("source_exact"),
        "source_precision": mean_key("source_precision"),
        "source_recall": mean_key("source_recall"),
        "source_f1": mean_key("source_f1"),
        "format_ok": mean_key("format_ok"),
        "answer_nonempty": mean_key("answer_nonempty"),
        "pred_ids_in_context": mean_key("pred_ids_in_context"),
        "pred_ids_in_corpus": mean_key("pred_ids_in_corpus"),
        "insufficient_empty_rate": mean_key("insufficient_empty"),
        "rouge_l_f1": mean_key("rouge_l_f1"),
        "n_examples": len(eval_records),
        "seconds": time.time() - start,
    }

    # Composite ưu tiên chọn đúng nguồn/chunk, sau đó format và chất lượng answer.
    metrics["composite_score"] = float(
        0.45 * metrics["source_f1"]
        + 0.20 * metrics["source_exact"]
        + 0.15 * metrics["format_ok"]
        + 0.10 * metrics["pred_ids_in_context"]
        + 0.10 * metrics["rouge_l_f1"]
    )

    return metrics, rows


# Records dùng cho generation eval.
eval_records_for_gen = eval_df.to_dict("records")
test_records_for_gen = test_df.to_dict("records")

print("Generation eval records:", len(eval_records_for_gen))
print("Generation test records:", len(test_records_for_gen))

Generation eval records: 50
Generation test records: 50


In [18]:
# Cell 17 — Callback: save best adapter by generation metrics + early stopping by composite score

class GenerationMetricBestSaverCallback(TrainerCallback):
    def __init__(
        self,
        tokenizer,
        eval_records: List[Dict[str, Any]],
        best_dir: str,
        max_examples: int = GEN_EVAL_MAX_EXAMPLES,
        every_n_evals: int = GEN_EVAL_EVERY_N_EVALS,
        patience: int = GEN_METRIC_PATIENCE,
        min_delta: float = GEN_METRIC_MIN_DELTA,
    ):
        self.tokenizer = tokenizer
        self.eval_records = eval_records
        self.best_dir = best_dir
        self.max_examples = max_examples
        self.every_n_evals = max(1, every_n_evals)
        self.patience = patience
        self.min_delta = min_delta

        self.eval_calls = 0
        self.best_score = -1.0
        self.bad_count = 0
        self.history = []

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None:
            return control

        self.eval_calls += 1
        if self.eval_calls % self.every_n_evals != 0:
            return control

        print("\nRunning generation metrics callback...")
        metrics, examples = compute_generation_metrics(
            model=model,
            tokenizer=self.tokenizer,
            eval_records=self.eval_records,
            max_examples=self.max_examples,
            verbose=False,
        )

        score = metrics["composite_score"]
        event = {
            "step": int(state.global_step),
            "epoch": float(state.epoch or 0.0),
            **metrics,
        }
        self.history.append(event)
        state.log_history.append({f"gen_{k}": v for k, v in event.items() if isinstance(v, (int, float))})

        print(json.dumps(event, ensure_ascii=False, indent=2))

        improved = score > (self.best_score + self.min_delta)

        if improved:
            self.best_score = score
            self.bad_count = 0

            if os.path.exists(self.best_dir):
                shutil.rmtree(self.best_dir)
            os.makedirs(self.best_dir, exist_ok=True)

            model.save_pretrained(self.best_dir)
            self.tokenizer.save_pretrained(self.best_dir)

            write_json(os.path.join(self.best_dir, "best_generation_metrics.json"), event)
            write_json(os.path.join(self.best_dir, "best_generation_examples.json"), examples[:20])
            write_json(os.path.join(self.best_dir, "generation_metric_history.json"), self.history)

            print(f"New best generation composite_score={score:.4f}. Saved adapter to: {self.best_dir}")
        else:
            self.bad_count += 1
            print(
                f"No improvement in generation composite_score. "
                f"best={self.best_score:.4f}, current={score:.4f}, bad_count={self.bad_count}/{self.patience}"
            )

        if self.bad_count >= self.patience:
            print("Generation metric early stopping triggered.")
            control.should_training_stop = True

        return control

In [19]:
# Cell 18 — Version-compatible TrainingArguments

def build_training_arguments() -> TrainingArguments:
    params = inspect.signature(TrainingArguments.__init__).parameters

    effective_eval_steps = int(EVAL_STEPS)
    effective_save_steps = int(EVAL_STEPS)

    if SAVE_STEPS != EVAL_STEPS:
        print(
            f"Warning: SAVE_STEPS={SAVE_STEPS} khác EVAL_STEPS={EVAL_STEPS}. "
            f"Sẽ dùng save_steps={effective_save_steps} để khớp eval_steps."
        )

    kwargs = dict(
        output_dir=OUTPUT_DIR,
        run_name=RUN_NAME,

        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",

        logging_steps=LOGGING_STEPS,

        save_strategy="steps",
        save_steps=effective_save_steps,
        save_total_limit=3,

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        bf16=USE_BF16,
        fp16=not USE_BF16,

        optim="paged_adamw_8bit",

        gradient_checkpointing=True,
        max_grad_norm=1.0,

        report_to=["tensorboard"],

        remove_unused_columns=False,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,

        seed=SEED,
    )

    if "eval_strategy" in params:
        kwargs["eval_strategy"] = "steps"
    elif "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = "steps"
    else:
        print("Warning: transformers version không có eval_strategy/evaluation_strategy.")
        kwargs["load_best_model_at_end"] = False
        kwargs.pop("metric_for_best_model", None)
        kwargs.pop("greater_is_better", None)

    if "eval_steps" in params:
        kwargs["eval_steps"] = effective_eval_steps

    if "do_eval" in params:
        kwargs["do_eval"] = True

    # Một số version không nhận key này.
    filtered = {k: v for k, v in kwargs.items() if k in params}
    dropped = sorted(set(kwargs) - set(filtered))
    if dropped:
        print("Dropped unsupported TrainingArguments keys:", dropped)

    args = TrainingArguments(**filtered)

    print("TrainingArguments created.")
    print("load_best_model_at_end:", getattr(args, "load_best_model_at_end", None))
    print("metric_for_best_model:", getattr(args, "metric_for_best_model", None))
    print("greater_is_better:", getattr(args, "greater_is_better", None))
    print("eval/save steps:", effective_eval_steps, effective_save_steps)

    return args


training_args = build_training_arguments()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TrainingArguments created.
load_best_model_at_end: True
metric_for_best_model: eval_loss
greater_is_better: False
eval/save steps: 50 50


In [20]:
# Cell 19 — Create trainer

gen_metric_callback = GenerationMetricBestSaverCallback(
    tokenizer=tokenizer,
    eval_records=eval_records_for_gen,
    best_dir=BEST_METRIC_ADAPTER_DIR,
    max_examples=GEN_EVAL_MAX_EXAMPLES,
    every_n_evals=GEN_EVAL_EVERY_N_EVALS,
    patience=GEN_METRIC_PATIENCE,
    min_delta=GEN_METRIC_MIN_DELTA,
)


def build_trainer() -> WeightedCETrainer:
    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=eval_during_train,
        data_collator=data_collator,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE,
                early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
            ),
            gen_metric_callback,
        ],
    )

    params = inspect.signature(WeightedCETrainer.__init__).parameters

    if "processing_class" in params:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in params:
        trainer_kwargs["tokenizer"] = tokenizer

    return WeightedCETrainer(**trainer_kwargs)


trainer = build_trainer()
print("Trainer created successfully.")

Trainer created successfully.


In [21]:
# Cell 20 — Sanity check one batch and active labels

batch = data_collator([tokenized["train"][0], tokenized["train"][1]])
print({k: tuple(v.shape) for k, v in batch.items()})

active = batch["labels"].ne(-100)
print("Active label tokens:", int(active.sum()))
print("Mean active loss weight:", float(batch["loss_weights"][active].mean()))

decoded_input = tokenizer.decode(batch["input_ids"][0], skip_special_tokens=False)
label_ids = batch["labels"][0][batch["labels"][0].ne(-100)]
decoded_labels = tokenizer.decode(label_ids, skip_special_tokens=False)

print("\nDecoded input preview:\n", decoded_input[:2000])
print("\nDecoded train labels preview:\n", decoded_labels[:1000])

{'input_ids': (2, 3128), 'attention_mask': (2, 3128), 'labels': (2, 3128), 'loss_weights': (2, 3128)}
Active label tokens: 185
Mean active loss weight: 1.233513355255127

Decoded input preview:
 <|im_start|>user
Câu hỏi:
Vì sao sau thắng lợi Bô Cô, quân Hậu Trần không lập tức lấy Đông Quan?

Tài liệu tham khảo:
[hf_wikipedia_nhà_hậu_trần_0002_98c9b7e2cef1] Nhà Hậu Trần
Tân Bình, đến tháng 6, 1408, Đặng Tất phá quân Thế Căng ở cửa biển Nhật Lệ, bắt giải đem về xử tử, nhà Hậu Trần làm chủ từ Nghệ An vào Thăng Hoa. Trận Bô Cô Tháng Chạp năm Mậu Tý (1408), Giản Định Đế hội tất cả quân Thuận Hóa, Tân Bình, Nghệ An, Diễn Châu, Thanh Hóa, rồi tiến ra đánh Đông Đô. Quân ra đến Trường An (Ninh Bình) thì các quan thuộc và những kẻ hào kiệt ở các nơi ra theo đông đảo. Quân nhà Minh đem tin ấy về báo cho vua Minh Thành Tổ biết. Chu Đệ sai Mộc Thạnh đem 40.000 quân ở Vân Nam sang đánh. Quân Minh cũng điều động 20.000 thủy quân tại Trung Quốc sẵn sàng sang tiếp chiến. Thêm vào đó, tiếp vận sứ Minh l

In [22]:
# Cell 21 — Train

train_result = trainer.train()

print("Train result:", train_result)
print("Best checkpoint by eval_loss:", trainer.state.best_model_checkpoint)
print("Best eval_loss:", trainer.state.best_metric)

trainer.save_state()

write_json(
    os.path.join(OUTPUT_DIR, "train_result.json"),
    {
        "train_result": train_result.metrics if hasattr(train_result, "metrics") else str(train_result),
        "best_model_checkpoint_by_eval_loss": trainer.state.best_model_checkpoint,
        "best_eval_loss": trainer.state.best_metric,
    },
)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,2.669996,0.298431
100,1.761843,0.288516
150,0.934587,0.318169
200,0.438666,0.391338
250,0.246027,0.413053



Running generation metrics callback...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{
  "step": 50,
  "epoch": 0.8888888888888888,
  "source_exact": 0.9,
  "source_precision": 0.9,
  "source_recall": 0.9,
  "source_f1": 0.9,
  "format_ok": 1.0,
  "answer_nonempty": 1.0,
  "pred_ids_in_context": 1.0,
  "pred_ids_in_corpus": 1.0,
  "insufficient_empty_rate": 0.4,
  "rouge_l_f1": 0.589026997920623,
  "n_examples": 50,
  "seconds": 947.1196849346161,
  "composite_score": 0.8939026997920624
}
New best generation composite_score=0.8939. Saved adapter to: /content/outputs/qwen2_5_3b_vnhistory_phase2_rag_qlora_best_by_generation_metric

Running generation metrics callback...


...

{
  "step": 250,
  "epoch": 4.391111111111111,
  "source_exact": 0.96,
  "source_precision": 0.96,
  "source_recall": 0.96,
  "source_f1": 0.96,
  "format_ok": 1.0,
  "answer_nonempty": 1.0,
  "pred_ids_in_context": 1.0,
  "pred_ids_in_corpus": 1.0,
  "insufficient_empty_rate": 1.0,
  "rouge_l_f1": 0.6282838620831813,
  "n_examples": 50,
  "seconds": 888.414781332016,
  "composite_score": 0.9368283862083181
}
No improvement in generation composite_score. best=0.9389, current=0.9368, bad_count=2/4

Running generation metrics callback...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{
  "step": 285,
  "epoch": 5.0,
  "source_exact": 0.94,
  "source_precision": 0.94,
  "source_recall": 0.94,
  "source_f1": 0.94,
  "format_ok": 1.0,
  "answer_nonempty": 1.0,
  "pred_ids_in_context": 1.0,
  "pred_ids_in_corpus": 1.0,
  "insufficient_empty_rate": 0.8,
  "rouge_l_f1": 0.6095180143637889,
  "n_examples": 50,
  "seconds": 896.101708650589,
  "composite_score": 0.9219518014363789
}
No improvement in generation composite_score. best=0.9389, current=0.9220, bad_count=3/4


Step,Training Loss,Validation Loss
50,2.669996,0.298431
100,1.761843,0.288516
150,0.934587,0.318169
200,0.438666,0.391338
250,0.246027,0.413053
285,0.226130,0.423512


Train result: TrainOutput(global_step=285, training_loss=1.719189501226994, metrics={'train_runtime': 9595.5251, 'train_samples_per_second': 0.469, 'train_steps_per_second': 0.03, 'total_flos': 2.1314476738176614e+17, 'train_loss': 1.719189501226994, 'epoch': 5.0})
Best checkpoint by eval_loss: /content/outputs/qwen2_5_3b_vnhistory_phase2_rag_qlora/checkpoint-100
Best eval_loss: 0.28851643204689026


In [35]:
# Cell 22 — Test-only eval WITHOUT trainer + safe batched Qwen generation
# Dùng được sau khi Cell reload best adapter đã del trainer.
# Cell này cũng override lại generate_from_user_text để các cell manual test phía sau bớt lỗi lặp user/dính prompt.

import os, json, time, re, random
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# =========================
# Config
# =========================

# A100: thử 8 trước. Nếu OOM thì giảm 4. Nếu dư VRAM thì thử 12 hoặc 16.
GEN_METRIC_BATCH_SIZE = 8

# None = chạy toàn bộ test set
TEST_GEN_MAX_EXAMPLES = None

# In vài mẫu để kiểm tra output thật
VERBOSE_EXAMPLES = 5

# Đúng yêu cầu: mặc định không lưu file.
# Muốn lưu metrics JSON thì đổi thành True.
SAVE_TEST_METRICS_JSON = False

# Có tính test loss thủ công không.
DO_TEST_LOSS = True

print("GEN_METRIC_BATCH_SIZE =", GEN_METRIC_BATCH_SIZE)
print("TEST_GEN_MAX_EXAMPLES =", TEST_GEN_MAX_EXAMPLES)
print("SAVE_TEST_METRICS_JSON =", SAVE_TEST_METRICS_JSON)


# =========================
# 1) Resolve model
# =========================

if "model" in globals():
    eval_model = model
    print("Using global model variable.")
elif "trainer" in globals():
    eval_model = trainer.model
    print("Using trainer.model.")
else:
    raise RuntimeError(
        "Không tìm thấy biến model hoặc trainer. "
        "Bạn cần chạy các cell load/reload model trước Cell 22."
    )

eval_model.eval()

old_use_cache = getattr(eval_model.config, "use_cache", None)
eval_model.config.use_cache = True

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

eval_model.config.pad_token_id = tokenizer.pad_token_id

if torch.cuda.is_available():
    torch.cuda.empty_cache()


def _get_model_device(m):
    try:
        return m.get_input_embeddings().weight.device
    except Exception:
        return next(m.parameters()).device


# =========================
# 2) Manual weighted test loss, không cần trainer
# =========================

@torch.inference_mode()
def compute_manual_weighted_lm_loss(
    model,
    dataset,
    collator,
    batch_size: int = 4,
) -> Dict[str, Any]:
    """
    Tính loss giống WeightedCETrainer:
    - chỉ tính token labels != -100
    - nhân loss_weights
    - không cần trainer
    """
    model.eval()
    device = _get_model_device(model)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collator,
    )

    total_weighted_loss = 0.0
    total_weight = 0.0
    total_active_tokens = 0
    t0 = time.time()

    for batch in tqdm(loader, desc="Manual weighted test loss"):
        loss_weights = batch.pop("loss_weights", None)

        labels = batch.pop("labels")
        input_ids = batch["input_ids"]
        attention_mask = batch.get("attention_mask", None)

        input_ids = input_ids.to(device)
        labels = labels.to(device)

        if attention_mask is not None:
            attention_mask = attention_mask.to(device)

        if loss_weights is None:
            loss_weights = labels.ne(-100).float()
        else:
            loss_weights = loss_weights.to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )

        logits = outputs.logits

        # Causal LM shift:
        # logits[:, t] predicts labels[:, t+1]
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        shift_weights = loss_weights[:, 1:].contiguous()

        vocab_size = shift_logits.size(-1)

        flat_logits = shift_logits.view(-1, vocab_size)
        flat_labels = shift_labels.view(-1)
        flat_weights = shift_weights.view(-1)

        token_loss = F.cross_entropy(
            flat_logits,
            flat_labels,
            reduction="none",
            ignore_index=-100,
        )

        active = flat_labels.ne(-100)

        if active.any():
            active_loss = token_loss[active]
            active_weights = flat_weights[active]

            total_weighted_loss += float((active_loss * active_weights).sum().detach().cpu())
            total_weight += float(active_weights.sum().detach().cpu())
            total_active_tokens += int(active.sum().detach().cpu())

    loss = total_weighted_loss / max(total_weight, 1.0)

    return {
        "test_manual_weighted_loss": loss,
        "test_active_tokens": total_active_tokens,
        "test_weight_sum": total_weight,
        "test_runtime_seconds": time.time() - t0,
        "test_samples": len(dataset),
        "test_batch_size": batch_size,
    }


if DO_TEST_LOSS:
    # Loss eval dùng batch riêng. Nếu OOM thì giảm 4.
    TEST_LOSS_BATCH_SIZE = min(8, GEN_METRIC_BATCH_SIZE)

    test_loss_metrics = compute_manual_weighted_lm_loss(
        model=eval_model,
        dataset=tokenized["test"],
        collator=data_collator,
        batch_size=TEST_LOSS_BATCH_SIZE,
    )

    print("\nManual weighted test loss metrics:")
    print(json.dumps(test_loss_metrics, ensure_ascii=False, indent=2))
else:
    test_loss_metrics = {
        "note": "Skipped manual test loss."
    }


# =========================
# 3) Safe Qwen generation
# =========================

def clean_qwen_generated_text(text: str) -> str:
    """
    Cắt output để tránh:
    - dính <|im_end|>
    - model sinh tiếp role user/system/assistant
    - lặp hội thoại
    """
    if text is None:
        return ""

    cut_markers = [
        IM_END,
        f"{IM_START}user",
        f"{IM_START}system",
        f"{IM_START}assistant",
        "\nuser\n",
        "\nUser\n",
        "\nUSER\n",
        "\nassistant\n",
        "\nAssistant\n",
    ]

    for marker in cut_markers:
        if marker and marker in text:
            text = text.split(marker)[0]

    special_tokens = [
        IM_START,
        IM_END,
        tokenizer.eos_token if tokenizer.eos_token else "",
        tokenizer.pad_token if tokenizer.pad_token else "",
    ]

    for tok in special_tokens:
        if tok:
            text = text.replace(tok, "")

    return normalize_text(text)


@torch.inference_mode()
def generate_from_user_texts_batched(
    model,
    tokenizer,
    user_texts: List[str],
    batch_size: int = 8,
    max_new_tokens: int = GEN_MAX_NEW_TOKENS,
    temperature: float = 0.0,
    top_p: float = 1.0,
) -> List[str]:
    """
    Batch generate cho Qwen chat format.
    Dùng left padding + stop ở <|im_end|>.
    """
    model.eval()
    device = _get_model_device(model)

    old_padding_side = getattr(tokenizer, "padding_side", "right")
    tokenizer.padding_side = "left"

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model.config.pad_token_id = tokenizer.pad_token_id

    im_end_id = tokenizer.convert_tokens_to_ids(IM_END)

    eos_ids = []
    if isinstance(tokenizer.eos_token_id, int) and tokenizer.eos_token_id >= 0:
        eos_ids.append(tokenizer.eos_token_id)

    if isinstance(im_end_id, int) and im_end_id >= 0 and im_end_id not in eos_ids:
        eos_ids.append(im_end_id)

    if not eos_ids:
        eos_ids = None

    outputs_text = []

    for start in tqdm(range(0, len(user_texts), batch_size), desc="Batched generation"):
        batch_user_texts = user_texts[start:start + batch_size]

        prompts = [
            build_prompt_from_user_text(user_text)
            for user_text in batch_user_texts
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Với batch + left padding, output luôn gồm padded prompt length + generated tokens.
        prompt_len = inputs["input_ids"].shape[1]

        gen_kwargs = dict(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
            repetition_penalty=1.05,
        )

        if temperature > 0:
            gen_kwargs["temperature"] = temperature
            gen_kwargs["top_p"] = top_p

        generated_ids = model.generate(**gen_kwargs)

        for i in range(len(batch_user_texts)):
            new_tokens = generated_ids[i][prompt_len:]
            raw_text = tokenizer.decode(new_tokens, skip_special_tokens=False)
            clean_text = clean_qwen_generated_text(raw_text)
            outputs_text.append(clean_text)

    tokenizer.padding_side = old_padding_side
    return outputs_text


# Override lại hàm cũ để các cell manual test phía sau dùng bản an toàn hơn.
@torch.inference_mode()
def generate_from_user_text(
    model,
    tokenizer,
    user_text: str,
    max_new_tokens: int = GEN_MAX_NEW_TOKENS,
    temperature: float = 0.0,
    top_p: float = 1.0,
) -> str:
    return generate_from_user_texts_batched(
        model=model,
        tokenizer=tokenizer,
        user_texts=[user_text],
        batch_size=1,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
    )[0]


# =========================
# 4) Batched generation metrics on test set
# =========================

def _safe_mean(values: List[float]) -> float:
    values = [v for v in values if v is not None]
    return float(sum(values) / len(values)) if values else 0.0


def compute_generation_metrics_batched_safe(
    model,
    tokenizer,
    eval_records: List[Dict[str, Any]],
    max_examples=None,
    batch_size: int = 8,
    verbose: bool = True,
) -> Tuple[Dict[str, float], List[Dict[str, Any]]]:

    records = list(eval_records)

    if max_examples is not None and len(records) > max_examples:
        rng = random.Random(SEED)
        records = rng.sample(records, max_examples)

    t0 = time.time()

    user_texts = [rec["user_text"] for rec in records]

    preds = generate_from_user_texts_batched(
        model=model,
        tokenizer=tokenizer,
        user_texts=user_texts,
        batch_size=batch_size,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        temperature=0.0,
        top_p=1.0,
    )

    rows = []
    metric_rows = []

    for idx, (rec, pred) in enumerate(zip(records, preds)):
        pred_ids = extract_source_ids_from_answer(pred)
        gold_ids = rec["gold_source_ids"]
        context_ids = rec["context_chunk_ids"]

        pred_answer_body = extract_answer_body(pred)
        gold_answer_body = rec["answer_body"]

        sm = set_metrics(pred_ids, gold_ids)

        format_ok = float(
            pred.strip().startswith("Nguồn được dùng:")
            and bool(ANSWER_SPLIT_RE.search(pred))
        )

        answer_nonempty = float(len(pred_answer_body.strip()) >= 10)
        pred_ids_in_context = float(all(cid in set(context_ids) for cid in pred_ids))
        pred_ids_in_corpus = float(all(cid in all_chunk_ids for cid in pred_ids))

        insufficient_empty = None
        if rec["type"] == "insufficient_context":
            insufficient_empty = float(pred_ids == [])

        metric_rows.append({
            **sm,
            "format_ok": format_ok,
            "answer_nonempty": answer_nonempty,
            "pred_ids_in_context": pred_ids_in_context,
            "pred_ids_in_corpus": pred_ids_in_corpus,
            "rouge_l_f1": rouge_l_f1(pred_answer_body, gold_answer_body),
            "insufficient_empty": insufficient_empty,
        })

        rows.append({
            "id": rec["id"],
            "type": rec["type"],
            "question": rec["question"],
            "gold_ids": gold_ids,
            "pred_ids": pred_ids,
            "context_chunk_ids": context_ids,
            "gold_answer": rec["assistant_text"],
            "pred_answer": pred,
        })

        if verbose and idx < VERBOSE_EXAMPLES:
            print("=" * 100)
            print("ID:", rec["id"])
            print("TYPE:", rec["type"])
            print("QUESTION:", rec["question"])
            print("GOLD IDS:", gold_ids)
            print("PRED IDS:", pred_ids)
            print("FORMAT OK:", format_ok)
            print("PRED ANSWER:")
            print(pred)

    metrics = {
        "source_exact": _safe_mean([m["source_exact"] for m in metric_rows]),
        "source_precision": _safe_mean([m["source_precision"] for m in metric_rows]),
        "source_recall": _safe_mean([m["source_recall"] for m in metric_rows]),
        "source_f1": _safe_mean([m["source_f1"] for m in metric_rows]),
        "format_ok": _safe_mean([m["format_ok"] for m in metric_rows]),
        "answer_nonempty": _safe_mean([m["answer_nonempty"] for m in metric_rows]),
        "pred_ids_in_context": _safe_mean([m["pred_ids_in_context"] for m in metric_rows]),
        "pred_ids_in_corpus": _safe_mean([m["pred_ids_in_corpus"] for m in metric_rows]),
        "insufficient_empty_rate": _safe_mean([
            m["insufficient_empty"]
            for m in metric_rows
            if m["insufficient_empty"] is not None
        ]),
        "rouge_l_f1": _safe_mean([m["rouge_l_f1"] for m in metric_rows]),
        "n_examples": len(records),
        "seconds": time.time() - t0,
        "batch_size": batch_size,
        "max_new_tokens": GEN_MAX_NEW_TOKENS,
    }

    metrics["composite_score"] = float(
        0.45 * metrics["source_f1"]
        + 0.20 * metrics["source_exact"]
        + 0.15 * metrics["format_ok"]
        + 0.10 * metrics["pred_ids_in_context"]
        + 0.10 * metrics["rouge_l_f1"]
    )

    return metrics, rows


print("\nGeneration metrics on test set:")

gen_start = time.time()

test_gen_metrics, test_gen_examples = compute_generation_metrics_batched_safe(
    model=eval_model,
    tokenizer=tokenizer,
    eval_records=test_records_for_gen,
    max_examples=TEST_GEN_MAX_EXAMPLES,
    batch_size=GEN_METRIC_BATCH_SIZE,
    verbose=True,
)

gen_seconds = time.time() - gen_start

print("\nTest generation metrics:")
print(json.dumps(test_gen_metrics, ensure_ascii=False, indent=2))
print(f"Generation test metrics finished in {gen_seconds:.2f} seconds")


# =========================
# 5) Optional save
# =========================

all_metrics = {
    "test_loss_metrics": test_loss_metrics,
    "test_generation_metrics": test_gen_metrics,
    "generation_runtime_seconds": gen_seconds,
    "model_source": "global model" if "model" in globals() else "trainer.model",
    "config": {
        "model_id": MODEL_ID,
        "phase1_adapter_dir": PHASE1_ADAPTER_DIR,
        "merged_base_dir": MERGED_BASE_DIR,
        "max_length": MAX_LENGTH,
        "source_line_loss_weight": SOURCE_LINE_LOSS_WEIGHT,
        "answer_loss_weight": ANSWER_LOSS_WEIGHT,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "gen_metric_batch_size": GEN_METRIC_BATCH_SIZE,
        "test_gen_max_examples": TEST_GEN_MAX_EXAMPLES,
        "gen_max_new_tokens": GEN_MAX_NEW_TOKENS,
    },
}

if "gen_metric_callback" in globals():
    all_metrics["best_generation_metric_callback"] = {
        "best_score": gen_metric_callback.best_score,
        "best_dir": BEST_METRIC_ADAPTER_DIR,
        "history": gen_metric_callback.history,
    }

if SAVE_TEST_METRICS_JSON:
    write_json(
        os.path.join(OUTPUT_DIR, "phase2_test_metrics_safe_no_trainer.json"),
        all_metrics,
    )

    write_json(
        os.path.join(OUTPUT_DIR, "test_generation_examples_safe_no_trainer.json"),
        test_gen_examples,
    )

    print("Saved test metrics to:", OUTPUT_DIR)
else:
    print("SAVE_TEST_METRICS_JSON=False, không lưu file JSON.")

# Restore use_cache
if old_use_cache is not None:
    eval_model.config.use_cache = old_use_cache

GEN_METRIC_BATCH_SIZE = 8
TEST_GEN_MAX_EXAMPLES = None
SAVE_TEST_METRICS_JSON = False
Using global model variable.


Manual weighted test loss:   0%|          | 0/7 [00:00<?, ?it/s]


Manual weighted test loss metrics:
{
  "test_manual_weighted_loss": 0.2983313242565031,
  "test_active_tokens": 5787,
  "test_weight_sum": 7060.7998046875,
  "test_runtime_seconds": 13.769608497619629,
  "test_samples": 50,
  "test_batch_size": 8
}

Generation metrics on test set:


Batched generation:   0%|          | 0/7 [00:00<?, ?it/s]

ID: sample_0013
TYPE: noisy_context
QUESTION: Khởi nghĩa Hương Khê kết thúc trong hoàn cảnh nào?
GOLD IDS: ['hf_wikipedia_phan_đình_phùng_0006_26a71d27e696']
PRED IDS: ['hf_wikipedia_phan_đình_phùng_0006_26a71d27e696']
FORMAT OK: 1.0
PRED ANSWER:
Nguồn được dùng: [hf_wikipedia_phan_đình_phùng_0006_26a71d27e696]

Trả lời:
Theo tài liệu, gần 3.000 quân do Nguyễn Thân cầm đầu ngày càng xiết chặt vòng vây. Trong trận giao tranh ác liệt ngày 28 tháng 12 năm 1895, Phan Đình Phùng bị thương nặng rồi hy sinh. Mười hai ngày sau khi ông hy sinh, Nguyễn Thân mới tới được núi Vũ Quang và núi Quạt; sau đó chính quyền Pháp và quan chức triều đình nhà Nguyễn đem quan tài về thôn Đông Thái để kiểm nghiệm.
ID: sample_0038
TYPE: insufficient_context
QUESTION: Tên đầy đủ của 3.000 giả tử do Dương Đình Nghệ nuôi là những ai?
GOLD IDS: []
PRED IDS: []
FORMAT OK: 1.0
PRED ANSWER:
Nguồn được dùng: []

Trả lời:
Tài liệu được cung cấp chỉ nêu Dương Đình Nghệ nuôi 3.000 giả tử, nhưng không nêu tên đầy đủ từng n

In [26]:
# Cell 23 — Export best phase2 adapter, zip, and copy to Google Drive
# Ưu tiên adapter tốt nhất theo generation composite metric.
# Nếu callback chưa từng save, fallback sang trainer.model hiện tại.

if os.path.exists(FINAL_EXPORT_DIR):
    shutil.rmtree(FINAL_EXPORT_DIR)
os.makedirs(FINAL_EXPORT_DIR, exist_ok=True)

source_best_dir = None

if os.path.exists(os.path.join(BEST_METRIC_ADAPTER_DIR, "adapter_config.json")):
    source_best_dir = BEST_METRIC_ADAPTER_DIR
    print("Using best adapter by generation metric:", source_best_dir)
    shutil.copytree(source_best_dir, FINAL_EXPORT_DIR, dirs_exist_ok=True)
else:
    print("No generation-best adapter found. Saving current trainer.model as final.")
    trainer.save_model(FINAL_EXPORT_DIR)
    tokenizer.save_pretrained(FINAL_EXPORT_DIR)

# Always save tokenizer and metadata.
tokenizer.save_pretrained(FINAL_EXPORT_DIR)

metadata = {
    "exported_at": datetime.now().isoformat(),
    "source_best_dir": source_best_dir,
    "final_export_dir": FINAL_EXPORT_DIR,
    "output_dir": OUTPUT_DIR,
    "phase1_adapter_dir": PHASE1_ADAPTER_DIR,
    "merged_base_dir": MERGED_BASE_DIR,
    "note": (
        "Phase2 adapter trained from a base model where phase1 LoRA was merged into Qwen2.5. "
        "This is a fresh LoRA adapter, not continued training from the phase1 LoRA checkpoint."
    ),
}

write_json(os.path.join(FINAL_EXPORT_DIR, "phase2_export_metadata.json"), metadata)

# Copy important metric files.
for fname in [
    "phase2_eval_test_metrics.json",
    "eval_generation_examples_current_model.json",
    "test_generation_examples_current_model.json",
    "train_result.json",
]:
    src = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(FINAL_EXPORT_DIR, fname))

print("\nFinal export folder:", FINAL_EXPORT_DIR)
print("Final export size MB:", get_size_mb(FINAL_EXPORT_DIR))

print("\nFiles in final export:")
for p in sorted(Path(FINAL_EXPORT_DIR).iterdir()):
    print(f"{p.name:45s} {get_size_mb(str(p)):10.2f} MB")

# Zip local.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_base_name = f"qwen_vnhistory_phase2_rag_best_adapter_{timestamp}"

local_zip_base = f"/content/outputs/{zip_base_name}"
local_zip_path = f"{local_zip_base}.zip"

if os.path.exists(local_zip_path):
    os.remove(local_zip_path)

shutil.make_archive(
    base_name=local_zip_base,
    format="zip",
    root_dir=FINAL_EXPORT_DIR,
)

print("\nCreated local zip:", local_zip_path)
print("Local zip size MB:", os.path.getsize(local_zip_path) / 1024 / 1024)

# Copy folder + zip to Drive.
drive_final_dir = os.path.join(DRIVE_BACKUP_DIR, "qwen_vnhistory_phase2_rag_best_adapter")
if os.path.exists(drive_final_dir):
    shutil.rmtree(drive_final_dir)
shutil.copytree(FINAL_EXPORT_DIR, drive_final_dir)

drive_zip_path = os.path.join(DRIVE_BACKUP_DIR, os.path.basename(local_zip_path))
shutil.copy2(local_zip_path, drive_zip_path)

print("\nCopied final adapter folder to Google Drive:")
print(drive_final_dir)
print("\nCopied final zip to Google Drive:")
print(drive_zip_path)
print("Drive zip size MB:", os.path.getsize(drive_zip_path) / 1024 / 1024)

Using best adapter by generation metric: /content/outputs/qwen2_5_3b_vnhistory_phase2_rag_qlora_best_by_generation_metric

Final export folder: /content/outputs/qwen_vnhistory_phase2_rag_best_adapter
Final export size MB: 239.36657905578613

Files in final export:
README.md                                           0.01 MB
adapter_config.json                                 0.00 MB
adapter_model.safetensors                         228.44 MB
best_generation_examples.json                       0.02 MB
best_generation_metrics.json                        0.00 MB
chat_template.jinja                                 0.00 MB
generation_metric_history.json                      0.00 MB
phase2_export_metadata.json                         0.00 MB
tokenizer.json                                     10.89 MB
tokenizer_config.json                               0.00 MB
train_result.json                                   0.00 MB

Created local zip: /content/outputs/qwen_vnhistory_phase2_rag_best_adapter

In [28]:
# Cell 24 — Reload best phase2 adapter for inference

RELOAD_BEST_FOR_INFERENCE = True

if RELOAD_BEST_FOR_INFERENCE:
    print("Cleaning training model from memory...")
    try:
        del trainer
    except Exception:
        pass
    try:
        del model
    except Exception:
        pass

    cleanup_cuda()

    base_infer = AutoModelForCausalLM.from_pretrained(
        MERGED_BASE_DIR,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=compute_dtype,
    )
    base_infer.config.use_cache = True

    model = PeftModel.from_pretrained(
        base_infer,
        FINAL_EXPORT_DIR,
        is_trainable=False,
    )
    model.eval()

    print("Loaded best phase2 adapter for inference:", FINAL_EXPORT_DIR)
else:
    model = trainer.model
    model.eval()
    model.config.use_cache = True
    print("Using current trainer.model for inference.")

Cleaning training model from memory...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loaded best phase2 adapter for inference: /content/outputs/qwen_vnhistory_phase2_rag_best_adapter


In [29]:
# Cell 25 — RAG inference helpers: retrieve chunks then generate answer

# Build a lightweight TF-IDF retriever from all_chunk_id.jsonl.
# Trong app thật, bạn có thể thay bằng FAISS/vector DB.
chunk_docs = []
chunk_ids_ordered = []
for c in chunks:
    cid = c.get("chunk_id", "")
    title = c.get("title", "")
    text = c.get("text", "")
    if cid and text:
        chunk_ids_ordered.append(cid)
        chunk_docs.append(normalize_text(f"{title}\n{text}"))

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=120_000,
    ngram_range=(1, 2),
)
tfidf_matrix = tfidf_vectorizer.fit_transform(chunk_docs)

print("TF-IDF matrix:", tfidf_matrix.shape)


def retrieve_chunks_tfidf(question: str, top_k: int = 5) -> List[Dict[str, Any]]:
    q_vec = tfidf_vectorizer.transform([normalize_text(question)])
    scores = linear_kernel(q_vec, tfidf_matrix).ravel()
    top_idx = np.argsort(scores)[::-1][:top_k]

    out = []
    for idx in top_idx:
        cid = chunk_ids_ordered[int(idx)]
        c = dict(chunk_by_id[cid])
        c["retrieval_score"] = float(scores[int(idx)])
        out.append(c)
    return out


def format_rag_user_prompt(question: str, context_chunks: List[Dict[str, Any]]) -> str:
    parts = [f"Câu hỏi:\n{normalize_text(question)}", "", "Tài liệu tham khảo:"]
    for c in context_chunks:
        cid = c.get("chunk_id", "")
        title = c.get("title", "")
        text = c.get("text", "")
        parts.append(f"[{cid}] {title}\n{normalize_text(text)}")
    return "\n".join(parts)


@torch.inference_mode()
def answer_with_rag(
    question: str,
    top_k: int = 5,
    context_chunks: Optional[List[Dict[str, Any]]] = None,
    max_new_tokens: int = 384,
    temperature: float = 0.0,
    top_p: float = 1.0,
) -> Dict[str, Any]:
    if context_chunks is None:
        context_chunks = retrieve_chunks_tfidf(question, top_k=top_k)

    user_text = format_rag_user_prompt(question, context_chunks)

    pred = generate_from_user_text(
        model=model,
        tokenizer=tokenizer,
        user_text=user_text,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
    )

    return {
        "question": question,
        "retrieved_context": [
            {
                "chunk_id": c.get("chunk_id"),
                "title": c.get("title"),
                "retrieval_score": c.get("retrieval_score"),
            }
            for c in context_chunks
        ],
        "prompt": user_text,
        "answer": pred,
        "used_chunk_ids": extract_source_ids_from_answer(pred),
    }


# Test nhanh với một sample trong test set.
sample = test_records_for_gen[0]
quick = answer_with_rag(
    question=sample["question"],
    context_chunks=[chunk_by_id[cid] for cid in sample["context_chunk_ids"]],
    max_new_tokens=256,
)
print("QUESTION:", quick["question"])
print("GOLD IDS:", sample["gold_source_ids"])
print("PRED IDS:", quick["used_chunk_ids"])
print("\nANSWER:\n", quick["answer"])

TF-IDF matrix: (520, 110637)
QUESTION: Khởi nghĩa Hương Khê kết thúc trong hoàn cảnh nào?
GOLD IDS: ['hf_wikipedia_phan_đình_phùng_0006_26a71d27e696']
PRED IDS: ['hf_wikipedia_phan_đình_phùng_0006_26a71d27e696']

ANSWER:
 Nguồn được dùng: [hf_wikipedia_phan_đình_phùng_0006_26a71d27e696]

Trả lời:
Theo tài liệu, khi quân Pháp ngày càng xiết chặt vòng vây, gần 3.000 quân do Nguyễn Thân cầm đầu khiến nghĩa quân phải rút lui. Ngày 28 tháng 12 năm 1895, Phan Đình Phùng hy sinh; mười hai ngày sau, Nguyễn Thân mới tới được núi Vũ Quang và núi Quạt.


In [30]:
# Cell 26 — Final manual tests: các câu phase1 đã từng inference thử
# Phase2 cần context, nên cell này sẽ retrieve top_k chunk từ all_chunk_id.jsonl rồi yêu cầu model chọn nguồn và trả lời.

manual_questions = [
    "Việc Lý Công Uẩn dời đô ra Thăng Long năm 1010 có ý nghĩa gì?",
    "Bình Ngô đại cáo ra đời trong bối cảnh nào?",
    "So sánh vai trò của nhà Lý và nhà Trần trong xây dựng và bảo vệ Đại Việt.",
    "Khởi nghĩa Lam Sơn diễn ra trong bối cảnh nào và kết quả ra sao?",
    "Chiến thắng Bạch Đằng năm 938 có ý nghĩa lịch sử gì?",
    "Nhà Trần đã làm gì để chống quân Nguyên Mông?",
    "Cải cách của Hồ Quý Ly có những điểm chính nào?",
    "Vì sao Quang Trung đại phá quân Thanh năm 1789 là sự kiện quan trọng?",
]

for q in manual_questions:
    result = answer_with_rag(
        question=q,
        top_k=5,
        max_new_tokens=384,
        temperature=0.0,
        top_p=1.0,
    )

    print("=" * 120)
    print("QUESTION:", q)
    print("\nRETRIEVED CHUNKS:")
    for c in result["retrieved_context"]:
        print(f"- {c['chunk_id']} | {c['title']} | score={c['retrieval_score']:.4f}")

    print("\nUSED CHUNK IDS:", result["used_chunk_ids"])
    print("\nANSWER:")
    print(result["answer"])

QUESTION: Việc Lý Công Uẩn dời đô ra Thăng Long năm 1010 có ý nghĩa gì?

RETRIEVED CHUNKS:
- hf_wikipedia_nhà_lý_0002_830b4c6c4cba | Nhà Lý | score=0.2184
- hf_wikipedia_nhà_lý_0001_de47f5602264 | Nhà Lý | score=0.1699
- hf_wikipedia_niên_biểu_lịch_sử_việt_nam_0001_43bf30f42b93 | Niên biểu lịch sử Việt Nam | score=0.0836
- hf_wikipedia_niên_biểu_lịch_sử_việt_nam_0001_43bf30f42b93 | Niên biểu lịch sử Việt Nam | score=0.0836
- hf_wikipedia_đại_cồ_việt_0004_52bff329f1ec | Đại Cồ Việt | score=0.0793

USED CHUNK IDS: []

ANSWER:
778 Nguyễn Nhạc xưng Hoàng đế, đặt niên hiệu Thái Đức, lập lên nhà Tây Sơn, đặt kinh đô tại Quy Nhơn Nguyễn Nhạc phong Nguyễn Huệ làm Bắc Bình Vương178519 tháng 1 – 20 tháng 1 Nguyễn Huệ phá tan quân Xiêm tại Rạch Gầm – Xoài Mút Nhà Tây Sơn 1788 Nguyễn Nhạc từ bỏ đế hiệu, chỉ xưng Tây Sơn vương22 tháng 12 Nguyễn Huệ xưng đế, đặt niên hiệu Quang Trung, đặt kinh đô tại Phú Xuân1789 Trận Ngọc Hồi – Đống Đa, đẩy lui quân xâm lược nhà Thanh, nhà Hậu Lê sụp đổ179216 tháng

## Gợi ý chỉnh nhanh nếu bị OOM

- Giảm `MAX_LENGTH` từ `4096` xuống `2048`.
- Giảm `PER_DEVICE_TRAIN_BATCH_SIZE` từ `2` xuống `1`.
- Tăng `GRADIENT_ACCUMULATION_STEPS` để giữ effective batch.
- Giảm `GEN_EVAL_MAX_EXAMPLES` từ `80` xuống `30` nếu generation callback quá chậm.
- Nếu merge phase1 adapter bị thiếu VRAM, hãy chạy bước merge trên GPU RAM cao hơn một lần, lưu `MERGED_BASE_DIR` vào Drive, rồi reload 4-bit để train phase2.